In [111]:
import json, math, pickle, re
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from typing import Optional, Dict, List

# RDKit
from rdkit import Chem
from rdkit.Chem import Descriptors
import joblib
from rdkit.Chem import AllChem, rdFingerprintGenerator

# Пути
BASE_DIR = Path("E:/Code/Poliforge/Polyforge-AI/Polyforge-AI")
BRIDGE_DIR = BASE_DIR / "data" / "configs" / "bridges"
BRIDGE_DIR.mkdir(parents=True, exist_ok=True)
DATA_RAW_DIR = BASE_DIR / "data" / "raw"
MODELS_DIR   = BASE_DIR / "models" / "nlp_model_bert"

FINGERPRINT_MODEL_PATH = BASE_DIR / "models" / "fingerprint_property_model.pkl"
artifacts = joblib.load(FINGERPRINT_MODEL_PATH)
lgb_models = artifacts['models']        # словарь: property -> LGBMRegressor
fp_scaler = artifacts['scaler']         # StandardScaler
fp_property_cols = artifacts['property_cols']

# Фингерпринт-генератор (один раз)
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module='sklearn')

def predict_properties_from_smiles(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = np.array(gen.GetFingerprint(mol)).reshape(1, -1)
    # Превращаем в DataFrame с именами колонок, чтобы не было предупреждения
    fp_df = pd.DataFrame(fp, columns=artifacts['feature_names'])
    preds_scaled = []
    for prop in artifacts['property_cols']:
        pred = artifacts['models'][prop].predict(fp_df)
        preds_scaled.append(pred[0])
    preds_scaled = np.array(preds_scaled).reshape(1, -1)
    preds = artifacts['scaler'].inverse_transform(preds_scaled).flatten()
    return dict(zip(artifacts['property_cols'], preds))

In [112]:
def check_smiles_valid(smiles: str) -> bool:
    """Может ли RDKit распарсить SMILES."""
    if not isinstance(smiles, str) or not smiles.strip():
        return False
    mol = Chem.MolFromSmiles(smiles)
    return mol is not None

def find_real_properties(smiles: str, df_real: pd.DataFrame, required_params: list) -> Optional[Dict[str, float]]:
    """
    Ищет SMILES в реальном датасете по колонке 'Polymer_SMILES'.
    Возвращает словарь фактических значений требуемых параметров.
    """
    match = df_real[df_real["Polymer_SMILES"] == smiles]
    if match.empty:
        return None
    row = match.iloc[0]
    props = {}
    for param in required_params:
        if param in row and pd.notna(row[param]):
            props[param] = float(row[param])
    return props if props else None

In [113]:
def load_nlp_requirements(path: Path) -> dict:
    with open(path, encoding='utf-8') as f:
        return json.load(f)

def prepare_requirement_strict(param: str, req: dict, thresholds: dict, is_user: bool) -> dict:
    """
    Ужесточает допустимый диапазон:
    Для пользовательского параметра: L = target - 0.05 * range, U = target + 0.05 * range
    Для авто: L = target - 0.15 * range, U = target + 0.15 * range
    range = thresholds[param]['max'] - thresholds[param]['min']
    sigma = 0.1 * range (для авто) или 0.05 * range (для пользователя)
    """
    t = thresholds.get(param)
    if t is None:
        # Нет порогов – используем консервативно
        L = req.get("min", req["target_value"])
        U = req.get("max", req["target_value"])
        sigma = (U - L) * 0.2 if U > L else 1.0
    else:
        full_range = t["max"] - t["min"]
        target = req["target_value"]
        if is_user:
            L = target - 0.02 * full_range   # жёстко: всего ±5% от размаха
            U = target + 0.02 * full_range
            sigma = 0.02 * full_range        # маленький масштаб для штрафа
        else:
            L = target - 0.10 * full_range   # авто – пошире
            U = target + 0.1 * full_range
            sigma = 0.05 * full_range

        # Не даём L/U вылезти за физические min/max
        L = max(L, t["min"])
        U = min(U, t["max"])

    return {
        "param": param,
        "target": req["target_value"],
        "L": L,
        "U": U,
        "sigma": sigma,
        "is_user": is_user,
        "category": req.get("category", "medium"),
        "extracted_name": req.get("extracted_name", param),
    }

# Загружаем мост
bridge = load_nlp_requirements(BRIDGE_DIR / "nlp_to_gen_bridge.json")
user_reqs = []
for p, r in bridge["user_specified"].items():
    preq = prepare_requirement_strict(p, r, THRESHOLDS, is_user=True)
    if preq:
        user_reqs.append(preq)

auto_reqs = []
for p, r in bridge["auto_filled"].items():
    preq = prepare_requirement_strict(p, r, THRESHOLDS, is_user=False)
    if preq:
        auto_reqs.append(preq)

all_reqs = user_reqs + auto_reqs
print(f"Ужесточённые требования готовы: {len(user_reqs)} польз., {len(auto_reqs)} авто")

Ужесточённые требования готовы: 4 польз., 33 авто


In [114]:
def find_smiles_column(df: pd.DataFrame) -> Optional[str]:
    """Находит колонку, содержащую SMILES/SELFIES/идентификатор молекулы."""
    for candidate in ["smiles", "Polymer_SMILES", "SELFIES"]:
        if candidate in df.columns:
            return candidate
    return None

def find_polymer_column(df: pd.DataFrame, param: str) -> Optional[str]:
    """Ищет колонку с фактическим значением параметра."""
    if param in df.columns:
        return param
    for suffix in ["_target", "_value"]:
        col = param + suffix
        if col in df.columns:
            return col
    return None

def load_polymers(path: Path, requirements: list) -> pd.DataFrame:
    """Загружает CSV из моста, оставляет нужные колонки, переименовывает ID в 'smiles'."""
    df = pd.read_csv(path)

    # 1. Находим колонку со SMILES и переименовываем в 'smiles'
    smile_col = find_smiles_column(df)
    if smile_col is None:
        raise KeyError("Не найдена колонка со SMILES/SELFIES в gen_to_val_bridge.csv")
    rename_map = {smile_col: "smiles"}

    # 2. Ищем колонки параметров
    for req in requirements:
        param = req["param"]
        col = find_polymer_column(df, param)
        if col:
            rename_map[col] = param

    # 3. Оставляем только нужные колонки
    keep_cols = ["smiles"]
    if "sample_index" in df.columns:
        keep_cols.append("sample_index")
    keep_cols += [k for k in rename_map if k != smile_col]  # все параметры
    keep_cols = list(dict.fromkeys(keep_cols))  # убираем дубликаты

    df = df[keep_cols].rename(columns=rename_map)
    return df

# Загружаем полимеры
polymers_df = load_polymers(BRIDGE_DIR / "gen_to_val_bridge.csv", all_reqs)
print(f"Загружено полимеров: {len(polymers_df)}")
print(f"Колонки: {list(polymers_df.columns)}")

Загружено полимеров: 5
Колонки: ['smiles', 'sample_index', 'Tm', 'rho', 'YM', 'LOI', 'Egc', 'Egb', 'Eib', 'CED', 'Ei', 'Eea', 'Eat', 'nc', 'ne', 'Xc', 'Xe', 'epse_6.0', 'epsc', 'epse_3.0', 'epse_1.78', 'epse_15.0', 'epse_4.0', 'epse_5.0', 'epse_2.0', 'epse_9.0', 'epse_7.0', 'epsb', 'TSb', 'TSy', 'permCH4', 'permCO2', 'permH2', 'permO2', 'permN2', 'permHe', 'Cp', 'Td', 'Tg']


In [115]:
def evaluate_parameter(polymer_value: Optional[float], req: dict) -> float:
    """Возвращает s в [0, 1]."""
    if polymer_value is None or (isinstance(polymer_value, float) and math.isnan(polymer_value)):
        # Для авто – нейтральные 0.5, для пользователя – жёсткий штраф 0.1 (не 0, чтобы не обнулить среднее)
        return 0.1 if req["is_user"] else 0.5

    L, U, sigma, target = req["L"], req["U"], req["sigma"], req["target"]
    if L <= polymer_value <= U:
        return 1.0

    dist = L - polymer_value if polymer_value < L else polymer_value - U
    score = math.exp(-0.5 * (dist / sigma) ** 2)
    return score

In [119]:
def compute_polymer_score(row: pd.Series, user_reqs: list, auto_reqs: list,
                          df_real: pd.DataFrame, sigma_rel=0.2) -> dict:
    smiles = row.get("smiles", "")
    if not check_smiles_valid(smiles):
        return {"overall_score": 0.0, "details": {}, "notes": "Invalid SMILES"}

    # 1. Базовая оценка соответствия требованиям NLP
    scores = {}
    for req in user_reqs + auto_reqs:
        param = req["param"]
        value = row.get(param)
        scores[param] = evaluate_parameter(value, req)

    # Среднее геометрическое с весами
    log_sum = 0.0
    total_weight = 0.0
    for req in user_reqs:
        w = 1.0
        s = max(scores[req["param"]], 1e-6)
        log_sum += w * math.log(s)
        total_weight += w
    for req in auto_reqs:
        w = 0.3
        s = max(scores[req["param"]], 1e-6)
        log_sum += w * math.log(s)
        total_weight += w
    base_score = math.exp(log_sum / total_weight) if total_weight > 0 else 0.0

    # 2. Штраф от БД
    db_penalty = 1.0
    required_params = [r["param"] for r in user_reqs + auto_reqs]
    real_props = find_real_properties(smiles, df_real, required_params)
    if real_props is not None:
        deviations = 0
        for param in required_params:
            if param in real_props and param in row:
                stated = row[param]
                if stated is None or math.isnan(stated):
                    continue
                actual = real_props[param]
                if actual != 0:
                    rel_diff = abs(stated - actual) / abs(actual)
                else:
                    rel_diff = abs(stated - actual)
                if rel_diff > 0.20:
                    deviations += 1
        db_penalty = max(0.5, 1.0 - 0.05 * deviations)

    # 3. Непрерывный реализм-штраф на основе LightGBM
    realism_score = 1.0
    pred_props = predict_properties_from_smiles(smiles)
    if pred_props is not None:
        factors = []
        for req in user_reqs + auto_reqs:
            param = req["param"]
            if param not in pred_props or param not in row:
                continue
            stated = row[param]
            if stated is None or math.isnan(stated):
                continue
            predicted = pred_props[param]
            if predicted == 0:
                continue
            rel_err = abs(stated - predicted) / abs(predicted)
            factor = math.exp(-0.5 * (rel_err / sigma_rel) ** 2)
            factor = max(factor, 1e-12)          # защита от 0
            factors.append(factor)

        if factors:
            log_prod = sum(math.log(f) for f in factors)
            realism_score = math.exp(log_prod / len(factors))
        else:
            realism_score = 1.0

    overall = base_score * db_penalty * realism_score

    notes = []
    if db_penalty < 1.0:
        notes.append(f"DB mismatch penalty: {db_penalty:.2f}")
    if realism_score < 0.999:
        notes.append(f"Realism: {realism_score:.4f}")

    return {
        "overall_score": overall,
        "details": scores,
        "db_penalty": db_penalty,
        "realism_score": realism_score,
        "notes": "; ".join(notes) if notes else "OK"
    }

In [124]:
def evaluate_all_polymers(df: pd.DataFrame, user_reqs: list, auto_reqs: list,
                          df_real: pd.DataFrame) -> pd.DataFrame:
    results = []
    for idx, row in df.iterrows():
        res = compute_polymer_score(row, user_reqs, auto_reqs, df_real)
        results.append({**row.to_dict(),
                        "score": res["overall_score"],
                        "realism_score": res.get("realism_score", 1.0),
                        "db_penalty": res.get("db_penalty", 1.0),
                        "score_details": res["details"],
                        "notes": res.get("notes", "")})
    result_df = pd.DataFrame(results)
    return result_df.sort_values("score", ascending=False)

evaluated_df = evaluate_all_polymers(polymers_df, user_reqs, auto_reqs, df_real)
print("Топ-5 полимеров:")
print(evaluated_df[["smiles", "score", "notes"]].head(5))

Топ-5 полимеров:
                                              smiles     score  \
4  [*]c1ccc(SC(=O)c2ccc(S(=O)(=O)c3ccc(-c4ccc5c(c...  0.698100   
3  [*]Nc1cc(-c2cc(C3CCC(N4C(=O)c5ccc(N6C(=O)c7ccc...  0.538465   
0  [*]Oc1cc([*])cc(-c2ccc(S(=O)(=O)c3ccc(N4C(=O)c...  0.022042   
1  [*]Oc1ccc2c(c1)C(=O)N(c1cccc(-c3ccc(S(=O)(=O)c...  0.017644   
2  [*]c1nc(-c2ccc(P(=O)(c3ccccc3)c3ccc(-c4ccc5c(c...  0.008802   

             notes  
4  Realism: 0.6981  
3  Realism: 0.5385  
0  Realism: 0.0220  
1  Realism: 0.0176  
2  Realism: 0.0088  


In [125]:
output_dir = BRIDGE_DIR
evaluated_df.to_csv(output_dir / "validation_full_results.csv", index=False)
top10 = evaluated_df.head(10)
top10.to_csv(output_dir / "validation_top.csv", index=False)

with open(output_dir / "summary.txt", "w", encoding="utf-8") as f:
    f.write(f"Оценено {len(evaluated_df)} полимеров\n")
    f.write(f"Запрос: {bridge['meta']['original_query']}\n\n")
    for i, (_, row) in enumerate(top10.iterrows()):
        f.write(f"{i+1}. {row['smiles']} – Score: {row['score']:.4f} ({row.get('notes', '')})\n")

print("Отчёты сохранены в", output_dir)

Отчёты сохранены в E:\Code\Poliforge\Polyforge-AI\Polyforge-AI\data\configs\bridges


In [126]:
test_smiles = "CC(C)(c1ccc(*)cc1)c1ccc(OCC(=O)OC(=O)c2cccc(c2)C(=O)OC(=O)CO*)cc1"
props = predict_properties_from_smiles(test_smiles)
print("Предсказанные свойства:", props)

Предсказанные свойства: {'Egc': np.float64(4.124098217782926), 'Egb': np.float64(3.7913736334092376), 'Eib': np.float64(3.5936963706010445), 'CED': np.float64(97.08735344827699), 'Ei': np.float64(5.926650249823525), 'Eea': np.float64(1.409086227445254), 'nc': np.float64(1.8411948746602058), 'ne': np.float64(1.57766184899455), 'Xc': np.float64(30.56333559172544), 'Xe': np.float64(29.427336995150984), 'epse_6.0': np.float64(3.5043099402684867), 'epsc': np.float64(4.257760426545684), 'epse_3.0': np.float64(3.812825709350227), 'epse_1.78': np.float64(4.013436735606909), 'epse_15.0': np.float64(2.512676095968456), 'epse_4.0': np.float64(3.726696086483013), 'epse_5.0': np.float64(3.652505646542651), 'epse_2.0': np.float64(3.934549498557509), 'epse_9.0': np.float64(3.077196928399939), 'epse_7.0': np.float64(3.366592073018857), 'epsb': np.float64(12.63766513414489), 'TSb': np.float64(86.28380502050848), 'TSy': np.float64(84.24976537176083), 'YM': np.float64(2466.9690772358126), 'permCH4': np.f

In [127]:
#df = pd.read_csv(output_dir / 'gen_to_val_bridge.csv')
header = list(df_real.columns)
print("Список имён столбцов:", header)

Список имён столбцов: ['ID', 'Name', 'SELFIES', 'Polymer_SMILES', 'Monomer_SMILES_1', 'Monomer_SMILES_2', 'Egc', 'Egb', 'Eib', 'CED', 'Ei', 'Eea', 'nc', 'ne', 'Xc', 'Xe', 'epse_6.0', 'epsc', 'epse_3.0', 'epse_1.78', 'epse_15.0', 'epse_4.0', 'epse_5.0', 'epse_2.0', 'epse_9.0', 'epse_7.0', 'epsb', 'TSb', 'TSy', 'YM', 'permCH4', 'permCO2', 'permH2', 'permO2', 'permN2', 'permHe', 'Cp', 'Td', 'Tg', 'Tm', 'rho', 'LOI']


In [134]:
# Тестовые SMILES из датасета
test_smiles_list = [
    '[*]c1ccc(OC(=O)N2C(=O)c3ccc(-c4ccc(C(=O)N(CC)c5ccc([*])cc5)cc4)cc3C2=O)cc1'
]

# Формируем DataFrame, где заявленные свойства = предсказанные
rows = []
for smi in test_smiles_list:
    preds = predict_properties_from_smiles(smi)
    if preds is None:
        continue
    row = {"smiles": smi}
    row.update(preds)          # все свойства как _target
    rows.append(row)

test_df = pd.DataFrame(rows)

# Прогоняем через валидатор
test_evaluated = evaluate_all_polymers(test_df, user_reqs, auto_reqs, df_real)

print("Тест полимеров из датасета (свойства = предсказания LightGBM):")
print(test_evaluated[["smiles", "score", "realism_score", "notes"]].head(5))

Тест полимеров из датасета (свойства = предсказания LightGBM):
                                              smiles     score  realism_score  \
0  [*]c1ccc(OC(=O)N2C(=O)c3ccc(-c4ccc(C(=O)N(CC)c...  0.079205            1.0   

  notes  
0    OK  
